# 08 - PyTorch Bridge

这本 `course` 不是压缩版提纲。课堂 notebook 会先把直觉、例子、代码和图跑通，再进入最后的 Predict / Modify 检查。

学习顺序：先读这一页的主线，遇到代码就运行；最后再做底部的小检查。真正写作业时进入同目录的 `homework.ipynb`。

In [ ]:
from pathlib import Path
import sys, math, random, inspect


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
PROJECT_ROOT = ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from micrograd.engine import Value
from micrograd.nn import Neuron, Layer, MLP

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ModuleNotFoundError:
    plt = None
    MATPLOTLIB_AVAILABLE = False

try:
    import torch
except ModuleNotFoundError:
    torch = None


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def assert_close(actual, expected, name='value', eps=1e-9):
    assert abs(actual - expected) < eps, f'{name}: expected {expected}, got {actual}'


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True

import importlib
import course.checks as course_checks
importlib.reload(course_checks)
qa_check = course_checks.qa_check


def show_svg(path):
    path = Path(path)
    try:
        from IPython.display import SVG, display
        display(SVG(filename=str(path)))
    except Exception:
        print('图已生成，但当前环境不能内嵌显示。请打开:', path.resolve())

print('project root:', ROOT)


In [ ]:
if torch is None:
    raise ModuleNotFoundError(
        '08_pytorch_bridge 开始 PyTorch 是必装依赖。请安装到 notebook 使用的 .venv，不要装到系统 Python：\n'
        '  .venv/bin/python -m pip install -r course/requirements-course.txt'
    )
print('torch version:', torch.__version__)


## 1. 概念对照表

| micrograd | PyTorch | 先这样理解 |
|---|---|---|
| `Value(2.0)` | `torch.tensor(2.0, requires_grad=True)` | 一个会记录梯度的数 |
| `.data` | `.item()` 或 tensor 本身 | 前向算出来的值 |
| `.grad` | `.grad` | 最终 loss 对它的导数 |
| `loss.backward()` | `loss.backward()` | 从 loss 反向传播 |
| `model.zero_grad()` | `optimizer.zero_grad()` 或手动清零 | 清空旧梯度 |
| 手写更新 `p.data += ...` | `optimizer.step()` | 用梯度更新参数 |

关键差别：micrograd 主要是一颗颗标量 `Value`；PyTorch 的 tensor 可以一次装很多数。

## 2. micrograd 版本：一个参数的一步训练

目标：让 `w * x` 接近 `y`。

In [ ]:
w = Value(2.0)
x = Value(3.0)
y = 7.0
learning_rate = 0.1

pred = w * x
loss = (pred - y) ** 2

print('pred:', pred.data)
print('loss:', loss.data)

loss.backward()
print('w.grad:', w.grad)

w.data += -learning_rate * w.grad
print('w after update:', w.data)

## 3. PyTorch 版本：同一件事

下面这段代码必须实际运行。进入这一节后，PyTorch 不再是可选项；它是把 micrograd 概念迁移到真实深度学习框架的硬门槛。

In [ ]:
w_t = torch.tensor(2.0, requires_grad=True)
x_t = torch.tensor(3.0)
y_t = torch.tensor(7.0)
learning_rate = 0.1

pred_t = w_t * x_t
loss_t = (pred_t - y_t) ** 2

print('pred:', pred_t.item())
print('loss:', loss_t.item())

loss_t.backward()
print('w.grad:', w_t.grad.item())

with torch.no_grad():
    w_t += -learning_rate * w_t.grad

print('w after update:', w_t.item())

In [ ]:
w_t = torch.tensor(2.0, requires_grad=True)
x_t = torch.tensor(3.0)
y_t = torch.tensor(7.0)
learning_rate = 0.1

pred_t = w_t * x_t
loss_t = (pred_t - y_t) ** 2

print('pred:', pred_t.item())
print('loss:', loss_t.item())

loss_t.backward()
print('w.grad:', w_t.grad.item())

with torch.no_grad():
    w_t += -learning_rate * w_t.grad
        
print('w after update:', w_t.item())

## 4. 为什么 PyTorch 更新时常见 `no_grad`

在 micrograd 里你直接改：

```python
p.data += -lr * p.grad
```

在 PyTorch 里，参数更新本身不应该被记录进下一张计算图，所以常见写法是：

```python
with torch.no_grad():
    p += -lr * p.grad
```

更常见的是让优化器做：

```python
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

## What To Remember

```text
1. 08 开始 PyTorch 是必装依赖，不再跳过。
2. micrograd 的 Value 可以看成 PyTorch Tensor 的标量玩具版。
3. 两边都有 data/value、grad、backward、zero_grad、update。
4. PyTorch Tensor 可以装向量、矩阵、批量数据。
5. PyTorch 更新参数时通常用 optimizer.step()。
6. 你理解 micrograd，PyTorch 的自动求导就不再是黑盒。
```

## 课堂检查：先预测，再改一行

这一段保留 `course` 的隐藏检查。你应该先自己填，再展开提示或答案。

## Predict - 跑通 PyTorch 同一小步


In [ ]:
# 填写说明：
# - 完成 student_torch_one_step，跑一遍 PyTorch 的 forward/loss/backward/update。
# - 运行本 cell，看 qa_check 的提示。

def student_torch_one_step():
    if torch is None:
        print('torch 未安装。先运行: .venv/bin/python -m pip install -r course/requirements-course.txt')
        return None
    # TODO 1: w = torch.tensor(2.0, requires_grad=True)
    # TODO 2: pred = w * x，其中 x=3, y=7
    # TODO 3: loss = (pred - y) ** 2，然后 backward
    # TODO 4: 在 torch.no_grad() 里更新 w: w += -0.1 * w.grad
    # TODO 5: 返回 pred.item(), loss.item(), w.grad.item(), w.item()
    pass


qa_check('qa_check_class_08_predict', globals())


<details><summary>Show / Hide 本题提示</summary>

- 这题要你跑出四个数字：预测值、loss、梯度、更新后的参数。
- `w.grad` 在 `loss.backward()` 后才会出现。
- 手动改 tensor 参数时，把更新放进 `with torch.no_grad():`。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_torch_one_step():
    w = torch.tensor(2.0, requires_grad=True)
    x = torch.tensor(3.0)
    y = torch.tensor(7.0)
    pred = w * x
    loss = (pred - y) ** 2
    loss.backward()
    grad_before_update = w.grad.item()
    with torch.no_grad():
        w += -0.1 * w.grad
    return pred.item(), loss.item(), grad_before_update, w.item()
```

</details>


## Run - micrograd 一步训练

In [ ]:
from micrograd.engine import Value

w = Value(2.0)
x = Value(3.0)
y = Value(7.0)
loss = (w*x-y)**2
loss.backward()
print('micrograd grad:', w.grad)
w.data -= 0.1*w.grad
print('updated w:', w.data)

## Run - 如果 torch 已安装，跑同一件事

In [ ]:
try:
    import torch
except Exception:
    torch = None
    print('torch 未安装，先按 course/requirements-course.txt 装本地 venv，不要全局安装。')

if torch is not None:
    w = torch.tensor(2.0, requires_grad=True)
    x = torch.tensor(3.0)
    y = torch.tensor(7.0)
    loss = (w*x-y)**2
    loss.backward()
    print('torch grad:', w.grad.item())
    with torch.no_grad():
        w -= 0.1*w.grad
    print('updated w:', w.item())

## Modify - zero_grad 是为什么？

In [ ]:
# 填写说明：
# - 完成 student_grad_accumulation_demo，亲手观察“不清 grad 会累加”。
# - 运行本 cell，看 qa_check 的提示。

def student_grad_accumulation_demo():
    if torch is None:
        print('torch 未安装。先运行: .venv/bin/python -m pip install -r course/requirements-course.txt')
        return None
    # TODO 1: 同一个 w 上 backward 两次，中间不要清 grad，记录 first_grad/second_grad。
    # TODO 2: w.grad.zero_() 后再 backward 一次，记录 after_zero_grad。
    pass


qa_check('qa_check_class_08_modify', globals())


<details><summary>Show / Hide 本题提示</summary>

- 用同一个 `w` 反复算 `loss = (w*3 - 7)**2`。
- 第一次 backward 后是 -6；第二次不清零会继续加一份。
- 清零后重新 backward，会回到本轮该有的梯度。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_grad_accumulation_demo():
    w = torch.tensor(2.0, requires_grad=True)
    x = torch.tensor(3.0)
    y = torch.tensor(7.0)

    loss = (w * x - y) ** 2
    loss.backward()
    first_grad = w.grad.item()

    loss = (w * x - y) ** 2
    loss.backward()
    second_grad = w.grad.item()

    w.grad.zero_()
    loss = (w * x - y) ** 2
    loss.backward()
    after_zero_grad = w.grad.item()
    return first_grad, second_grad, after_zero_grad
```

</details>
